In [ ]:
"""
AI-Based Early Sepsis Detection and Prediction — IMPROVED VERSION v2
=====================================================================
Changes vs v1:
  FIX 1 — _build_features() now uses pd.concat() instead of iterative
           column insertion.  Eliminates hundreds of PerformanceWarnings
           and ~30-50% RAM overhead from DataFrame fragmentation.
  FIX 2 — Train sequences use a streaming tf.data generator instead of
           materialising the full X_tr array.  Reduces peak RAM from ~8 GB
           to <1 GB during sequence creation (critical for T4 Colab).
  FIX 3 — Rolling windows reduced from [4,8,12] to [4,8].  Cuts feature
           count from 272 to ~190, reducing sequence tensor size by ~30%.
  FIX 4 — max_neg_per_patient lowered (30 to 20) for tighter class balance.

Kept from v1:
  Focal Loss, GRU + Multi-Head Attention, fixed asymmetric sampling,
  Platt scaling calibration, F2-score threshold tuning.
"""

import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, roc_auc_score, average_precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, GRU, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D, Add, Conv1D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision

# ──────────────────────────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────────────────────────
CSV_PATH       = "Sepsis Prediction Dataset.csv"
TARGET         = "SepsisLabel"
GROUP          = "Patient_ID"
HOUR_COL       = "Hour"

N_STEPS            = 24
PREDICTION_WINDOW  = 6

TEST_SIZE      = 0.20
VAL_SIZE       = 0.10
RANDOM_STATE   = 42

BATCH_SIZE     = 128
EPOCHS         = 40            # more headroom; early stopping handles the rest
PATIENCE       = 8             # was 5; val_roc_auc is slower to move than pr_auc
LEARNING_RATE  = 3e-4          # lower LR: slows overfitting, smoother convergence

NEGATIVE_MULTIPLIER  = 3
MAX_COL_MISSING      = 0.98
MAX_TRAIN_SAMPLES    = 200_000  # cap on NEGATIVES only; all positives always included
SHUFFLE_BUFFER       = 15_000
MAX_NEG_PER_PATIENT  = 20

FOCAL_GAMMA    = 2.0
FOCAL_ALPHA    = 0.85          # was 0.75; push harder on false negatives


# ──────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ──────────────────────────────────────────────────────────────────────────────
def set_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ──────────────────────────────────────────────────────────────────────────────
# Focal Loss
# ──────────────────────────────────────────────────────────────────────────────
def focal_loss(gamma: float = FOCAL_GAMMA, alpha: float = FOCAL_ALPHA):
    def loss_fn(y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce     = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha  + (1 - y_true) * (1 - alpha)
        fl      = alpha_t * tf.pow(1.0 - p_t, gamma) * bce
        return tf.reduce_mean(fl)
    return loss_fn


# ──────────────────────────────────────────────────────────────────────────────
# Causal Preprocessor  — FIX 1: uses pd.concat, not iterative column insertion
# ──────────────────────────────────────────────────────────────────────────────
class CausalPreprocessor:
    """
    Rolling windows: [4, 8]  (FIX 3 — was [4, 8, 12]).
    Feature count is now ~190 instead of 272.
    """

    ROLL_WINDOWS = [4, 8]

    def __init__(self, target_col=TARGET, group_col=GROUP,
                 hour_col=HOUR_COL, max_col_missing=MAX_COL_MISSING):
        self.target_col      = target_col
        self.group_col       = group_col
        self.hour_col        = hour_col
        self.max_col_missing = max_col_missing
        self.base_cols    = None
        self.feature_cols = None
        self.medians      = None
        self.scaler       = None

    def _sort(self, df):
        return df.sort_values([self.group_col, self.hour_col]).reset_index(drop=True)

    def _build_features(self, df):
        """
        All new columns assembled in ONE pd.concat call.
        No iterative df[col] = ... inside loops → zero fragmentation.
        """
        grp        = df.groupby(self.group_col, sort=False)
        new_frames = []

        # First differences
        diffs = grp[self.base_cols].diff().fillna(0.0).astype(np.float32)
        diffs.columns = [f"{c}_diff" for c in self.base_cols]
        new_frames.append(diffs)

        # Rolling mean + std
        for w in self.ROLL_WINDOWS:
            roll  = grp[self.base_cols].rolling(window=w, min_periods=1)
            rmean = roll.mean().reset_index(level=0, drop=True).astype(np.float32)
            rstd  = roll.std(ddof=0).fillna(0.0).reset_index(level=0, drop=True).astype(np.float32)
            rmean.columns = [f"{c}_rmean{w}" for c in self.base_cols]
            rstd.columns  = [f"{c}_rstd{w}"  for c in self.base_cols]
            new_frames.append(rmean)
            new_frames.append(rstd)

        extra = pd.concat(new_frames, axis=1)
        return pd.concat([df, extra], axis=1)

    def fit(self, df_train):
        df       = self._sort(df_train.copy())
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        exclude  = {self.target_col, self.group_col, self.hour_col}
        candidate = [c for c in num_cols if c not in exclude]

        miss_rate      = df[candidate].isna().mean()
        self.base_cols = [c for c in candidate if miss_rate[c] <= self.max_col_missing]
        if not self.base_cols:
            raise ValueError("No usable feature columns after missing-rate filtering.")

        for c in self.base_cols:
            df[c] = df.groupby(self.group_col, sort=False)[c].ffill()
        self.medians = df[self.base_cols].median(numeric_only=True)
        df[self.base_cols] = df[self.base_cols].fillna(self.medians)

        df = self._build_features(df)

        diff_cols  = [f"{c}_diff"      for c in self.base_cols]
        roll_cols  = ([f"{c}_rmean{w}" for c in self.base_cols for w in self.ROLL_WINDOWS] +
                      [f"{c}_rstd{w}"  for c in self.base_cols for w in self.ROLL_WINDOWS])
        scale_cols = self.base_cols + diff_cols + roll_cols

        self.feature_cols = scale_cols
        self.scaler       = StandardScaler()
        self.scaler.fit(df[scale_cols].astype(np.float32))
        return self

    def transform(self, df):
        if self.scaler is None:
            raise RuntimeError("Preprocessor not fitted.")
        out = self._sort(df.copy())
        for c in self.base_cols:
            if c not in out.columns:
                out[c] = np.nan
            out[c] = out.groupby(self.group_col, sort=False)[c].ffill()
        out[self.base_cols] = out[self.base_cols].fillna(self.medians)
        out = self._build_features(out)
        out[self.feature_cols] = self.scaler.transform(
            out[self.feature_cols].astype(np.float32)
        ).astype(np.float32)
        if self.target_col in out.columns:
            out[self.target_col] = out[self.target_col].fillna(0).astype(np.int32)
        return out


# ──────────────────────────────────────────────────────────────────────────────
# Sequence builder — unified for train / val / test
#
# SPEED FIX: All splits materialised as float16 (halves RAM vs float32).
# The tf.data pipeline is .cache()'d — epoch 2+ reads from RAM, no Python
# re-iteration overhead, ~10-20x faster per epoch.
#
# BALANCE FIX: max_samples caps ONLY negative sequences. All positive
# (septic) sequences are always included. The old approach of breaking
# early on combined iteration starved the train set of positives whenever
# septic patients appeared late in groupby order — causing total model
# collapse (zero recall, zero F1).
# ──────────────────────────────────────────────────────────────────────────────
def _extract_patient_sequences(g, feature_cols, hour_col, target_col,
                               n_steps, prediction_window,
                               max_neg_per_patient, dtype):
    """Extract all sequences for one patient. Returns (list_of_(window,y), is_septic)."""
    g     = g.sort_values(hour_col)
    feats = g[feature_cols].to_numpy(dtype=np.float32)
    labs  = g[target_col].to_numpy(dtype=np.int32)

    onset_arr = np.where(labs == 1)[0]
    onset_idx = int(onset_arr[0]) if onset_arr.size > 0 else None

    if onset_idx is None:
        n = len(g)
        t_indices = (np.linspace(0, n - 1, max_neg_per_patient, dtype=int)
                     if n > max_neg_per_patient else np.arange(n))
    else:
        t_indices = np.arange(min(onset_idx + 1, len(g)))

    seqs = []
    for t in t_indices:
        start  = max(0, t - n_steps + 1)
        window = feats[start: t + 1]
        if window.shape[0] < n_steps:
            pad    = np.repeat(window[:1], n_steps - window.shape[0], axis=0)
            window = np.vstack([pad, window])
        y = 0
        if onset_idx is not None:
            delta = onset_idx - t
            y = 1 if 1 <= delta <= prediction_window else 0
        seqs.append((window.astype(dtype), np.int32(y)))
    return seqs, onset_idx is not None


def create_sequences(
    df, feature_cols,
    n_steps=N_STEPS, prediction_window=PREDICTION_WINDOW,
    target_col=TARGET, group_col=GROUP, hour_col=HOUR_COL,
    max_neg_per_patient=MAX_NEG_PER_PATIENT,
    max_samples=None,   # cap on NEGATIVES only for train; None = no cap (val/test)
    dtype=np.float16,   # float16 for train (RAM); float32 for val/test
    rng_seed=RANDOM_STATE,
):
    """
    Build (X, y) arrays with guaranteed positive representation.

    Strategy (train, max_samples set):
      1. Collect ALL septic patient sequences — never capped.
      2. Fill remaining budget (max_samples - n_pos) with randomly sampled
         negative sequences from non-septic patients.
      3. Shuffle combined list so positives are not front-loaded.

    Val/test (max_samples=None): collect everything as before.
    """
    pos_seqs, neg_seqs = [], []
    rng = np.random.RandomState(rng_seed)

    for pid, g in df.groupby(group_col, sort=False):
        seqs, is_septic = _extract_patient_sequences(
            g, feature_cols, hour_col, target_col,
            n_steps, prediction_window, max_neg_per_patient, dtype,
        )
        if is_septic:
            pos_seqs.extend(seqs)
        else:
            neg_seqs.extend(seqs)

    # Cap only negatives so positives are always fully represented
    if max_samples is not None and len(neg_seqs) > 0:
        neg_budget = max(0, max_samples - len(pos_seqs))
        if len(neg_seqs) > neg_budget:
            idx      = rng.choice(len(neg_seqs), neg_budget, replace=False)
            neg_seqs = [neg_seqs[i] for i in idx]

    combined = pos_seqs + neg_seqs
    perm     = rng.permutation(len(combined))
    combined = [combined[i] for i in perm]

    if not combined:
        raise RuntimeError("create_sequences produced 0 sequences — check your data split.")

    X = np.asarray([s[0] for s in combined], dtype=dtype)
    y = np.asarray([s[1] for s in combined], dtype=np.int32)

    pos    = int((y == 1).sum())
    neg    = int((y == 0).sum())
    ram_mb = X.nbytes / 1024 / 1024
    print(f"  Sequences: {len(y):,} | Pos: {pos:,} | Neg: {neg:,} | "
          f"Pos ratio: {pos / max(1, len(y)):.4f} | RAM: {ram_mb:.0f} MB")
    if pos == 0:
        raise RuntimeError("No positive sequences — check data splits or PREDICTION_WINDOW.")
    return X, y


def make_dataset(X, y, batch_size=BATCH_SIZE, shuffle=False, cache=False):
    """
    Build a tf.data pipeline.  Pipeline order matters for correctness & speed:

      from_tensor_slices   — X stays float16 in RAM
      [.cache()]           — pin individual float16 samples (cheaper than f32)
      [.shuffle()]         — AFTER cache so every epoch gets a fresh shuffle
      .batch()             — group into batches
      .map(cast f16→f32)   — promote just one batch at a time (~2 MB) to f32
      .prefetch()          — overlap GPU compute with next batch preparation

    Putting shuffle BEFORE cache (old ordering) meant epoch 2+ always read
    samples in the same fixed order — defeating the point of shuffling.
    """
    ds = tf.data.Dataset.from_tensor_slices((X, y))  # float16 throughout
    if cache:
        ds = ds.cache()          # cache raw samples; enables fresh shuffle each epoch
    if shuffle:
        ds = ds.shuffle(min(SHUFFLE_BUFFER, len(X)), seed=RANDOM_STATE,
                        reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False)
    # Cast per-batch: only the current batch (~2 MB) is float32 at any moment
    ds = ds.map(lambda x, lbl: (tf.cast(x, tf.float32), lbl),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


# ──────────────────────────────────────────────────────────────────────────────
# Positive oversampling with noise augmentation
# ──────────────────────────────────────────────────────────────────────────────
def oversample_positives(
    X, y,
    target_ratio=0.25,   # desired positive fraction after oversampling
    noise_std=0.02,      # small Gaussian noise so duplicates aren't identical
    rng_seed=RANDOM_STATE,
):
    """
    Duplicate positive sequences (with small additive noise) until they make up
    target_ratio of the training set.

    With only 5% positives the model barely encounters them per batch (~6 per
    128-sample batch) and never builds strong positive-class representations
    before overfitting.  Boosting to 25% means ~32 positives per batch.

    noise_std=0.02 prevents exact-duplicate memorisation while preserving the
    clinical signal in the sequence.

    class_weight is removed from model.fit() — it was redundant with focal loss
    and double-penalising negatives was destabilising the gradient.
    """
    rng      = np.random.RandomState(rng_seed)
    pos_idx  = np.where(y == 1)[0]
    neg_cnt  = int((y == 0).sum())
    cur_ratio = len(pos_idx) / max(1, len(y))

    if cur_ratio >= target_ratio:
        print(f"  Oversample: already at {cur_ratio:.3f} >= target {target_ratio:.3f}, skipping.")
        return X, y

    # pos_needed so that pos/(pos+neg) == target_ratio
    target_pos   = int(neg_cnt * target_ratio / (1 - target_ratio))
    extra_needed = max(0, target_pos - len(pos_idx))

    samp_idx = rng.choice(pos_idx, size=extra_needed, replace=True)
    X_extra  = X[samp_idx].astype(np.float32)
    X_extra += rng.normal(0, noise_std, X_extra.shape).astype(np.float32)
    X_extra  = X_extra.astype(X.dtype)
    y_extra  = np.ones(extra_needed, dtype=y.dtype)

    X_new = np.concatenate([X, X_extra], axis=0)
    y_new = np.concatenate([y, y_extra], axis=0)
    perm  = rng.permutation(len(y_new))
    X_new, y_new = X_new[perm], y_new[perm]

    print(f"  Oversample: {len(pos_idx):,} -> {int(y_new.sum()):,} positives | "
          f"ratio {cur_ratio:.3f} -> {y_new.mean():.3f} | "
          f"Total: {len(y_new):,} | RAM: {X_new.nbytes/1024/1024:.0f} MB")
    return X_new, y_new


# ──────────────────────────────────────────────────────────────────────────────
# Model
# ──────────────────────────────────────────────────────────────────────────────
def build_gru_attention_model(input_shape):
    inp = Input(shape=input_shape, name="input")
    x   = Conv1D(64, kernel_size=3, padding="causal",
                 activation="relu", name="conv1d")(inp)
    x   = LayerNormalization()(x)

    # recurrent_dropout disabled: it forces slow non-cuDNN path on GPU
    gru1     = GRU(128, return_sequences=True, dropout=0.20,
                   name="gru1")(x)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=32,
                                  dropout=0.15, name="mha")(gru1, gru1)
    attn_out = LayerNormalization()(Add()([gru1, attn_out]))

    gru2   = GRU(64, return_sequences=True, dropout=0.20, name="gru2")(attn_out)
    gru2   = LayerNormalization()(gru2)
    pooled = GlobalAveragePooling1D(name="gap")(gru2)

    # L2 regularisation on Dense layers to combat train/val AUROC gap
    reg = tf.keras.regularizers.l2(1e-4)
    x   = Dense(64, activation="relu", kernel_regularizer=reg)(pooled)
    x   = Dropout(0.40)(x)
    x   = Dense(32, activation="relu", kernel_regularizer=reg)(x)
    x   = Dropout(0.30)(x)
    out = Dense(1, activation="sigmoid", dtype="float32", name="output")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
        loss=focal_loss(),
        metrics=[
            tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
            tf.keras.metrics.AUC(curve="PR",  name="pr_auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


# ──────────────────────────────────────────────────────────────────────────────
# Calibration
# ──────────────────────────────────────────────────────────────────────────────
def calibrate_probabilities(val_probs, val_labels, test_probs):
    """
    Platt scaling via logistic regression.
    Guard: if the calibrated mean is < 1/3 of the raw mean, the calibrator
    is collapsing all predictions toward zero (happens when model discrimination
    is weak and val positives are scarce). In that case skip calibration and
    return raw probs unchanged.
    """
    lr = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs")
    lr.fit(val_probs.reshape(-1, 1), val_labels)
    cal = lr.predict_proba(test_probs.reshape(-1, 1))[:, 1]
    raw_mean = test_probs.mean()
    cal_mean = cal.mean()
    print(f"  Calibration: prob mean {raw_mean:.4f} -> {cal_mean:.4f}")
    if cal_mean < raw_mean / 3.0:
        print("  WARNING: calibrator collapsed predictions toward zero — "
              "skipping calibration, using raw probabilities.")
        return test_probs.copy(), None
    return cal, lr


# ──────────────────────────────────────────────────────────────────────────────
# Threshold tuning (F2 — recall-weighted for clinical safety)
# ──────────────────────────────────────────────────────────────────────────────
def tune_threshold_f2(y_true, probs):
    best_th, best = 0.5, -1.0
    for th in np.linspace(0.10, 0.70, 121):
        score = fbeta_score(y_true, (probs >= th).astype(int), beta=2, zero_division=0)
        if score > best:
            best, best_th = score, float(th)
    return best_th, best


# ──────────────────────────────────────────────────────────────────────────────
# Metrics
# ──────────────────────────────────────────────────────────────────────────────
def summarize_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {
        "Threshold":             threshold,
        "Accuracy":              accuracy_score(y_true, pred),
        "Precision":             precision_score(y_true, pred, zero_division=0),
        "Recall":                recall_score(y_true, pred, zero_division=0),
        "F1":                    f1_score(y_true, pred, zero_division=0),
        "F2":                    fbeta_score(y_true, pred, beta=2, zero_division=0),
        "AUROC":                 roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else float("nan"),
        "AUPRC":                 average_precision_score(y_true, probs) if len(np.unique(y_true)) > 1 else float("nan"),
        "PredictedPositiveRate": float(pred.mean()),
        "ProbMean":              float(probs.mean()),
        "ProbStd":               float(probs.std()),
    }


# ──────────────────────────────────────────────────────────────────────────────
# Live simulation
# ──────────────────────────────────────────────────────────────────────────────
def simulate_patient_live(
    patient_id, raw_df, preprocessor, model, calibrator,
    feature_cols, n_steps=N_STEPS, prediction_window=PREDICTION_WINDOW,
    threshold=0.5, group_col=GROUP, hour_col=HOUR_COL,
    target_col=TARGET, smooth_window=3,
):
    patient_raw = raw_df[raw_df[group_col] == patient_id].copy()
    if patient_raw.empty:
        print(f"Patient {patient_id} not found.")
        return pd.DataFrame()

    patient_raw  = patient_raw.sort_values(hour_col).reset_index(drop=True)
    patient_proc = preprocessor.transform(patient_raw.copy())

    feats       = patient_proc[feature_cols].to_numpy(dtype=np.float32)
    true_labels = patient_raw[target_col].fillna(0).astype(np.int32).to_numpy()
    hours       = patient_raw[hour_col].to_numpy()

    onset_arr  = np.where(true_labels == 1)[0]
    onset_idx  = int(onset_arr[0]) if onset_arr.size > 0 else None
    onset_hour = hours[onset_idx]  if onset_idx is not None else None

    raw_risks, future_labels = [], []
    print(f"\n--- Live Simulation: Patient {patient_id} ---")

    for t in range(len(feats)):
        start = max(0, t - n_steps + 1)
        seq   = feats[start: t + 1]
        if seq.shape[0] < n_steps:
            pad = np.repeat(seq[:1], n_steps - seq.shape[0], axis=0)
            seq = np.vstack([pad, seq])
        raw_risks.append(float(model.predict(seq[None], verbose=0)[0, 0]))
        y_fut = 0
        if onset_idx is not None:
            delta = onset_idx - t
            y_fut = 1 if 1 <= delta <= prediction_window else 0
        future_labels.append(y_fut)

    raw_arr   = np.array(raw_risks, dtype=np.float32)
    cal_risks = (calibrator.predict_proba(raw_arr.reshape(-1, 1))[:, 1]
                 if calibrator is not None else raw_arr.copy())

    a = 2.0 / (smooth_window + 1)
    smoothed    = np.zeros_like(cal_risks)
    smoothed[0] = cal_risks[0]
    for i in range(1, len(cal_risks)):
        smoothed[i] = a * cal_risks[i] + (1 - a) * smoothed[i - 1]

    for t in range(len(feats)):
        print(f"Hour {int(hours[t]):3d} | Raw {raw_risks[t]:.3f} | "
              f"Cal {smoothed[t]:.3f} | "
              f"Fut{prediction_window}h {future_labels[t]} | "
              f"{'*** ALERT ***' if smoothed[t] >= threshold else 'OK'}")

    sim_df = pd.DataFrame({
        "Hour": hours, "RawRisk": raw_risks, "CalibratedRisk": cal_risks,
        "SmoothedRisk": smoothed, "FutureWindowLabel": future_labels,
        "CurrentSepsisLabel": true_labels,
    })

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.fill_between(sim_df["Hour"], 0, sim_df["SmoothedRisk"], alpha=0.15)
    ax.plot(sim_df["Hour"], sim_df["SmoothedRisk"],
            label="Smoothed Calibrated Risk", linewidth=2.5)
    ax.plot(sim_df["Hour"], sim_df["CalibratedRisk"],
            linestyle="--", alpha=0.5, linewidth=1, label="Calibrated Risk (raw)")
    ax.axhline(threshold, color="orange", linestyle="--",
               linewidth=1.5, label=f"Threshold ({threshold:.2f})")
    pos_mask = np.array(future_labels) == 1
    if pos_mask.any():
        ax.scatter(sim_df["Hour"][pos_mask], sim_df["SmoothedRisk"][pos_mask],
                   marker="x", s=80, zorder=5,
                   label=f"True +ve (next {prediction_window}h)")
    if onset_hour is not None:
        ax.axvline(onset_hour, color="red", linestyle=":",
                   linewidth=2, label=f"Sepsis Onset (hour {int(onset_hour)})")
    ax.set_ylim(0, 1.02)
    ax.set_title(f"Live Sepsis Risk Simulation — Patient {patient_id}", fontsize=14)
    ax.set_xlabel("Hour"); ax.set_ylabel("Risk Probability")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.savefig(f"simulation_patient_{patient_id}.png", dpi=150)
    plt.show()

    print(f"\nSummary | mean={smoothed.mean():.4f} std={smoothed.std():.4f} "
          f"min={smoothed.min():.4f} max={smoothed.max():.4f}")
    return sim_df


# ──────────────────────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────────────────────
def main():
    set_seed(RANDOM_STATE)
    tf.keras.backend.clear_session()
    gc.collect()

    gpus = tf.config.experimental.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            mixed_precision.set_global_policy("mixed_float16")
            print(f"GPU(s) detected: {len(gpus)} — mixed_float16 enabled")
        except Exception as e:
            print("GPU config error:", e)
    else:
        print("No GPU. Running on CPU.")

    print("Loading data...")
    df = pd.read_csv(
        CSV_PATH, low_memory=False,
        dtype={GROUP: np.int32, TARGET: np.int8, HOUR_COL: np.int16},
        on_bad_lines='skip',   # skip rows with wrong number of fields
    )
    print(f"  (Malformed rows skipped automatically if any)")
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    df = df.dropna(subset=[GROUP, TARGET, HOUR_COL]).copy()
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype(np.float32)
    df[GROUP]  = df[GROUP].astype(np.int32)
    df[TARGET] = df[TARGET].astype(np.int8)
    df = df.sort_values([GROUP, HOUR_COL]).reset_index(drop=True)
    print(f"Rows: {len(df):,} | Cols: {df.shape[1]} | Patients: {df[GROUP].nunique():,}")
    print(f"Positive rate: {df[TARGET].mean():.4f}")

    # Patient-level balance
    ps          = df.groupby(GROUP)[TARGET].max()
    septic_ids  = ps[ps == 1].index.values
    ns_ids      = ps[ps == 0].index.values
    print(f"\nSeptic: {len(septic_ids)} | Non-septic: {len(ns_ids)}")
    n_keep  = min(len(ns_ids), NEGATIVE_MULTIPLIER * len(septic_ids))
    keep_ns = np.random.RandomState(RANDOM_STATE).choice(ns_ids, n_keep, replace=False)
    df      = df[df[GROUP].isin(np.concatenate([septic_ids, keep_ns]))].reset_index(drop=True)
    print(f"Patients kept: {df[GROUP].nunique():,} | Rows: {len(df):,} | "
          f"Pos rate: {df[TARGET].mean():.4f}")

    # Splits
    gss = GroupShuffleSplit(1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    trv_idx, tst_idx = next(gss.split(df, groups=df[GROUP].values))
    df_trv = df.iloc[trv_idx].reset_index(drop=True)
    df_test_raw = df.iloc[tst_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(1, test_size=VAL_SIZE / (1 - TEST_SIZE),
                             random_state=RANDOM_STATE)
    tr_idx, val_idx = next(gss2.split(df_trv, groups=df_trv[GROUP].values))
    df_tr_raw  = df_trv.iloc[tr_idx].reset_index(drop=True)
    df_val_raw = df_trv.iloc[val_idx].reset_index(drop=True)
    print(f"\nPatients | train={df_tr_raw[GROUP].nunique()} "
          f"val={df_val_raw[GROUP].nunique()} "
          f"test={df_test_raw[GROUP].nunique()}")

    # Preprocessing — silent, fast, no fragmentation warnings
    print("\nFitting preprocessor...")
    prep = CausalPreprocessor()
    prep.fit(df_tr_raw)
    feature_cols = prep.feature_cols
    print(f"Feature count: {len(feature_cols)}")

    print("Transforming splits...")
    df_tr   = prep.transform(df_tr_raw)
    df_val  = prep.transform(df_val_raw)
    df_test = prep.transform(df_test_raw)
    gc.collect()

    # ── Sequences ─────────────────────────────────────────────────────────────
    # MEMORY ORDERING: create one split → hand it to tf.data (which makes its
    # own internal copy) → immediately del the numpy array → next split.
    # This keeps peak RAM to ~1 split at a time instead of all 3 at once.
    # All arrays are float16; make_dataset casts to float32 lazily per-batch.

    print(f"\nCreating train sequences (float16, capped at {MAX_TRAIN_SAMPLES:,})...")
    X_tr, y_tr = create_sequences(
        df_tr, feature_cols,
        max_samples=MAX_TRAIN_SAMPLES,
        dtype=np.float16,
    )
    if y_tr.sum() == 0:
        raise RuntimeError("No positive samples in train sequences.")

    # Oversample positives to 25% — forces ~32 positive samples per 128-sample batch
    # (up from ~6). The model must learn positive-class representations early or it
    # converges to a constant-output prior before generalisable features emerge.
    print("Oversampling train positives...")
    X_tr, y_tr = oversample_positives(X_tr, y_tr, target_ratio=0.25)

    train_ds = make_dataset(X_tr, y_tr, shuffle=True, cache=True)
    del X_tr; gc.collect()
    del df_tr; gc.collect()

    print("Creating val sequences...")
    X_val, y_val = create_sequences(df_val, feature_cols, dtype=np.float16)
    if y_val.sum() == 0:
        raise RuntimeError("No positive samples in val sequences.")
    val_ds = make_dataset(X_val, y_val, shuffle=False, cache=True)
    del X_val; gc.collect()
    del df_val; gc.collect()

    print("Creating test sequences...")
    X_test, y_test = create_sequences(df_test, feature_cols, dtype=np.float16)
    test_ds = make_dataset(X_test, y_test, shuffle=False, cache=False)
    del X_test; gc.collect()
    del df_test; gc.collect()

    # Class weights REMOVED: oversampling + focal loss together are sufficient.
    # Using class_weight ON TOP of focal loss double-penalises negatives, causing
    # gradient instability and collapsing outputs toward a constant prediction.

    # ── Build & train model ────────────────────────────────────────────────────
    model = build_gru_attention_model((N_STEPS, len(feature_cols)))
    model.summary()

    callbacks = [
        # Watch val_roc_auc: far more stable than val_pr_auc under severe
        # class imbalance.  val_pr_auc spiked at epoch 1 by chance and never
        # recovered, causing premature stopping after just 6 epochs.
        EarlyStopping(monitor="val_roc_auc", mode="max", patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        # Halve LR after 4 epochs without val_roc_auc improvement
        ReduceLROnPlateau(monitor="val_roc_auc", mode="max", factor=0.5,
                          patience=4, min_lr=1e-6, verbose=1),
    ]
    del df_tr_raw; gc.collect()  # df_val_raw already freed above

    print(f"\nTraining (max {EPOCHS} epochs, early stop patience={PATIENCE})...")
    print("Note: epoch 1 is slower (cache fill); epochs 2+ will be much faster.\n")
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        callbacks=callbacks, verbose=1,
    )

    # Calibrate
    print("\nCalibrating...")
    val_raw  = model.predict(val_ds,  verbose=0).ravel()
    test_raw = model.predict(test_ds, verbose=0).ravel()
    cal_test, calibrator = calibrate_probabilities(val_raw, y_val, test_raw)
    # Use same calibrator (or None) for val threshold tuning
    if calibrator is not None:
        cal_val = calibrator.predict_proba(val_raw.reshape(-1, 1))[:, 1]
    else:
        cal_val = val_raw.copy()

    # Threshold
    best_thresh, best_f2 = tune_threshold_f2(y_val, cal_val)
    print(f"\nBest threshold (F2): {best_thresh:.3f}  F2={best_f2:.4f}")

    # Evaluate
    metrics = summarize_metrics(y_test, cal_test, threshold=best_thresh)
    print("\n=== Test Metrics ===")
    for k, v in metrics.items():
        print(f"  {k:>25}: {v:.4f}" if isinstance(v, float) else f"  {k:>25}: {v}")

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(test_raw,  bins=60, alpha=0.7, label="Raw")
    axes[0].set_title("Raw Prob Distribution")
    axes[1].hist(cal_test,  bins=60, alpha=0.7, color="orange")
    axes[1].axvline(best_thresh, color="red", linestyle="--",
                    label=f"Threshold {best_thresh:.2f}")
    axes[1].set_title("Calibrated Prob Distribution")
    for ax in axes: ax.legend()
    plt.tight_layout(); plt.savefig("probability_distributions.png", dpi=150); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history["loss"],     label="Train")
    axes[0].plot(history.history["val_loss"], label="Val")
    axes[0].set_title("Focal Loss"); axes[0].legend()
    axes[1].plot(history.history["pr_auc"],     label="Train")
    axes[1].plot(history.history["val_pr_auc"], label="Val")
    axes[1].set_title("AUPRC"); axes[1].legend()
    plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

    # Simulation
    septic_pats = []
    for pid, g in df_test_raw.groupby(GROUP):
        onset = g[g[TARGET] == 1].sort_values(HOUR_COL)
        if len(onset) > 0:
            septic_pats.append((pid, int(onset.iloc[0][HOUR_COL])))

    if not septic_pats:
        print("No septic patients in test split.")
        return

    good = [(p, o) for p, o in septic_pats if 6 <= o <= 18]
    sel_id, sel_onset = good[0] if good else septic_pats[0]
    print(f"\nSimulating patient {sel_id} (onset hour {sel_onset})")

    sim_df = simulate_patient_live(
        patient_id=sel_id, raw_df=df, preprocessor=prep,
        model=model, calibrator=calibrator, feature_cols=feature_cols,
        threshold=best_thresh,
    )
    sim_df.to_csv(f"simulation_patient_{sel_id}.csv", index=False)
    model.save("sepsis_gru_attention_model.keras")
    print("Done.")


if __name__ == "__main__":
    main()